# Embeddings visuales y textuales con CLIP

Este notebook reutiliza el preprocesamiento del notebook 03 y extrae las representaciones visuales y textuales de CLIP. El encoder se importa desde `src/` y, al estar congelado, sus embeddings pueden cachearse para el entrenamiento posterior del modelo VLA.

## 1. Configuración e imports

Se definen las variables editables y se instancia `CLIPEncoder`. La arquitectura y la carga de CLIP permanecen en `src/clip_encoder.py`; aquí solo se crea el objeto y se usan sus interfaces públicas.

In [10]:
from pathlib import Path
import json
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Permitimos importar los módulos del proyecto cuando se ejecuta desde notebooks/.
ROOT_DIR = Path(os.getcwd()).resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.clip_encoder import CLIPEncoder
from src.dataset import VLADataset, mapear_indices_globales

# Variables editables del experimento.
RUTA = Path(r'E:\TFM_datasets\taco_play_lerobot')
NOMBRE_MODELO_CLIP = 'ViT-B-32'
PESOS_PREENTRENADOS = 'openai'
CONGELAR_CLIP = True
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 4
NORMALIZAR_EMBEDDINGS = True
CACHEAR_EMBEDDINGS = True
RUTA_CACHE = ROOT_DIR / 'data' / 'cache_embeddings'

# El encoder expone preprocess y tokenizer para construir el Dataset.
encoder = CLIPEncoder(NOMBRE_MODELO_CLIP, PESOS_PREENTRENADOS, DISPOSITIVO, CONGELAR_CLIP)
print(f'Modelo CLIP: {NOMBRE_MODELO_CLIP} ({PESOS_PREENTRENADOS})')
print(f'Dispositivo: {DISPOSITIVO}')

c:\TFM_Codigo\TFM-VLA\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Modelo CLIP: ViT-B-32 (openai)
Dispositivo: cpu


## 2. Carga de datos y creación de los `DataLoader`

Se cargan las particiones procesadas si existen. Como respaldo, se reconstruyen con la misma lógica de 03. Los rangos de normalización se leen del JSON generado por 03 y no se recalculan.

Cada `index` global se convierte en una ruta de MP4 y un frame local usando los limites fisicos de los videos. Esto evita depender de timestamps que pueden no coincidir exactamente con los frames codificados.

In [11]:
RUTA_PROCESADO = ROOT_DIR / 'data' / 'procesado_clip'
RUTA_PARAMETROS = ROOT_DIR / 'data' / 'parametros_normalizacion.json'
nombres = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}

if all((RUTA_PROCESADO / nombre).exists() for nombre in nombres.values()) and all(
    "index" in pd.read_parquet(RUTA_PROCESADO / nombre).columns
    for nombre in nombres.values()
):
    # Ruta rápida: reutiliza las tablas persistidas.
    particiones = {clave: pd.read_parquet(RUTA_PROCESADO / nombre) for clave, nombre in nombres.items()}
    print(f'Particiones cargadas desde {RUTA_PROCESADO}')
else:
    # Respaldo si 03 aún no ha guardado sus tablas.
    tareas = pd.read_parquet(RUTA / 'meta' / 'tasks.parquet')
    mapa_tareas = dict(zip(tareas['task_index'], tareas.index))
    archivos = sorted((RUTA / 'data').rglob('*.parquet'))
    datos = pd.concat([pd.read_parquet(a, columns=['action','next.done','episode_index','frame_index','index','task_index']) for a in archivos], ignore_index=True)
    datos['accion'] = list(np.column_stack((datos['next.done'].to_numpy(dtype=np.float32), np.vstack(datos['action'].to_numpy()).astype(np.float32))))
    # El indice global se asigna a los limites fisicos de los MP4.
    rutas_video, frames_locales = mapear_indices_globales(
        datos['index'].to_numpy(),
        RUTA / 'videos' / 'observation.images.rgb_static',
    )
    muestras = datos.copy()
    muestras['instruccion'] = muestras['task_index'].map(mapa_tareas)
    muestras['imagen'] = rutas_video
    muestras['fotograma_video'] = frames_locales
    tabla_final = muestras[['imagen','fotograma_video','instruccion','accion','episode_index','frame_index','index','task_index']]
    # La división es por episodios para evitar fuga entre particiones.
    rng = np.random.default_rng(42)
    episodios_unicos = tabla_final['episode_index'].drop_duplicates().to_numpy().copy(); rng.shuffle(episodios_unicos)
    n_train = int(0.80 * len(episodios_unicos)); n_val = n_train + int(0.10 * len(episodios_unicos))
    grupos = {'train': set(episodios_unicos[:n_train]), 'validation': set(episodios_unicos[n_train:n_val]), 'test': set(episodios_unicos[n_val:])}
    particiones = {clave: tabla_final[tabla_final['episode_index'].isin(ep)].reset_index(drop=True) for clave, ep in grupos.items()}
    RUTA_PROCESADO.mkdir(parents=True, exist_ok=True)
    for clave, tabla in particiones.items(): tabla.to_parquet(RUTA_PROCESADO / nombres[clave], index=False)
    print(f'Tablas reconstruidas y guardadas en {RUTA_PROCESADO}')

if not RUTA_PARAMETROS.exists():
    raise FileNotFoundError(f'No existe {RUTA_PARAMETROS}. Ejecuta primero el notebook 03.')
with open(RUTA_PARAMETROS, encoding='utf-8') as archivo:
    parametros = json.load(archivo)
minimo_accion = np.asarray(parametros['minimo'], dtype=np.float32)
escala_accion = np.asarray(parametros['escala'], dtype=np.float32)

def normalizar_accion(accion):
    # Usa exclusivamente los rangos de train guardados por el notebook 03.
    return (np.asarray(accion, dtype=np.float32) - minimo_accion) / escala_accion

tablas = {clave: tabla.copy() for clave, tabla in particiones.items()}
for tabla in tablas.values():
    tabla['imagen'] = tabla['imagen'].astype(str)
    tabla['accion'] = tabla['accion'].apply(normalizar_accion)
datasets = {clave: VLADataset(tabla, encoder.preprocess, encoder.tokenizer) for clave, tabla in tablas.items()}
loaders = {'train': DataLoader(datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=0), 'validation': DataLoader(datasets['validation'], batch_size=BATCH_SIZE, shuffle=False, num_workers=0), 'test': DataLoader(datasets['test'], batch_size=BATCH_SIZE, shuffle=False, num_workers=0)}
print({clave: len(ds) for clave, ds in datasets.items()})
print(f'Batch size: {BATCH_SIZE}')

66 muestras trasladadas de file-000.mp4 a file-001.mp4
Tablas reconstruidas y guardadas en C:\TFM_Codigo\TFM-VLA\data\procesado_clip
{'train': 190212, 'validation': 23760, 'test': 23826}
Batch size: 4


## 3. Embedding visual

Se toma un lote y se pasa al encoder visual. El resultado contiene un vector por imagen; con `ViT-B/32` la dimensión esperada es 512.

In [12]:
imagenes_batch, tokens_batch, acciones_batch = next(iter(loaders['train']))
embeddings_imagen = encoder.encode_image(imagenes_batch)
print(f'Entrada visual: {tuple(imagenes_batch.shape)}')
print(f'Embedding visual: {tuple(embeddings_imagen.shape)}')
print('Cada fila representa una imagen en el espacio multimodal de CLIP.')
assert embeddings_imagen.shape[0] == imagenes_batch.shape[0]
if NOMBRE_MODELO_CLIP == 'ViT-B-32':
    assert embeddings_imagen.shape[1] == 512
    print('La dimensión 512 coincide con ViT-B/32.')

Entrada visual: (4, 3, 224, 224)
Embedding visual: (4, 512)
Cada fila representa una imagen en el espacio multimodal de CLIP.
La dimensión 512 coincide con ViT-B/32.


## 4. Embedding textual

Se reutilizan los tokens del mismo lote. La dimensión coincide con la visual porque ambas modalidades se proyectan al mismo espacio de CLIP.

In [13]:
embeddings_texto = encoder.encode_text(tokens_batch)
print(f'Entrada textual: {tuple(tokens_batch.shape)}')
print(f'Embedding textual: {tuple(embeddings_texto.shape)}')
assert embeddings_texto.shape == embeddings_imagen.shape
print('Las dimensiones visual y textual coinciden.')

Entrada textual: (4, 77)
Embedding textual: (4, 512)
Las dimensiones visual y textual coinciden.


## 5. Comprobación conjunta: similitud coseno

Se compara cada imagen con su instrucción y, opcionalmente, con un texto desplazado dentro del lote. La media de los pares correctos debería ser superior si el lote contiene instrucciones informativas.

In [14]:
imagenes_norm = F.normalize(embeddings_imagen.float(), dim=-1)
textos_norm = F.normalize(embeddings_texto.float(), dim=-1)
similitudes_correctas = (imagenes_norm * textos_norm).sum(dim=-1)
if len(textos_norm) > 1:
    # El desplazamiento crea pares negativos sencillos sin cambiar el lote.
    textos_aleatorios = textos_norm.roll(shifts=1, dims=0)
    similitudes_aleatorias = (imagenes_norm * textos_aleatorios).sum(dim=-1)
    print(f'Media pares correctos:  {similitudes_correctas.mean().item():.4f}')
    print(f'Media pares aleatorios: {similitudes_aleatorias.mean().item():.4f}')
else:
    print(f'Similitud del único par: {similitudes_correctas.item():.4f}')

Media pares correctos:  0.2298
Media pares aleatorios: 0.2258


## 6. Cacheo de embeddings

Como CLIP está congelado, sus salidas no cambian entre épocas. Se recorren todos los lotes y se guarda un `.npz` por partición con embeddings visuales, textuales y acciones para ahorrar cómputo en el transformer.

In [15]:
def extraer_embeddings(loader):
    """Extrae los embeddings de todos los lotes conservando su correspondencia."""
    imagenes, textos, acciones = [], [], []
    for imagenes_batch, tokens_batch, acciones_batch in loader:
        imagenes.append(encoder.encode_image(imagenes_batch).detach().cpu())
        textos.append(encoder.encode_text(tokens_batch).detach().cpu())
        acciones.append(acciones_batch.cpu())
    emb_imagenes = torch.cat(imagenes); emb_textos = torch.cat(textos)
    if NORMALIZAR_EMBEDDINGS:
        emb_imagenes = F.normalize(emb_imagenes.float(), dim=-1)
        emb_textos = F.normalize(emb_textos.float(), dim=-1)
    return emb_imagenes.numpy(), emb_textos.numpy(), torch.cat(acciones).numpy()

if CACHEAR_EMBEDDINGS:
    RUTA_CACHE.mkdir(parents=True, exist_ok=True)
    for nombre, loader in loaders.items():
        imagenes, textos, acciones = extraer_embeddings(loader)
        salida = RUTA_CACHE / f'{nombre}.npz'
        np.savez_compressed(salida, imagenes=imagenes, textos=textos, acciones=acciones)
        print(f'{nombre}: {imagenes.shape[0]:,} muestras guardadas en {salida}')
else:
    print('CACHEAR_EMBEDDINGS=False: se omite el recorrido completo.')

KeyboardInterrupt: 

## 7. Resumen

Cada muestra queda representada por un embedding visual, uno textual en el mismo espacio multimodal y su acción normalizada.

In [ ]:
print('Pipeline CLIP completado correctamente.')
print(f'Dimensión visual: {embeddings_imagen.shape[-1]}')
print(f'Dimensión textual: {embeddings_texto.shape[-1]}')
print(f'Cache activada: {CACHEAR_EMBEDDINGS}')